### Setup

- Locates project and data/raw/ folder so the notebook can open the JSON file.

In [1]:
from pathlib import Path
import sys

# Locate the project root from either supported working directory.
PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "src" / "soccer_predictor").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src" / "soccer_predictor").is_dir():
    raise FileNotFoundError(
        "Start the notebook from the project root or notebooks folder."
    )

# Allow imports from src/soccer_predictor.
source_path = str(PROJECT_ROOT / "src")

if source_path not in sys.path:
    sys.path.insert(0, source_path)

# Locate the season JSON files.
DATA_DIRECTORY = PROJECT_ROOT / "data" / "raw"

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIRECTORY)
import json
import pandas as pd

Project root: C:\Users\arenk\OneDrive\Desktop\soccer-predictor
Data directory: C:\Users\arenk\OneDrive\Desktop\soccer-predictor\data\raw


### Load and Seperate

- Converts the JSON matches into a DataFrame, parses dates, and checks which fixtures have two valid goal totals.
- Completed fixtures go into test_matches, sorted chronologically; the others go into pending_matches.
- Only completed matches get goal columns.

In [2]:
filename = DATA_DIRECTORY / "premier_league_2026_27.json"

with open(filename, encoding="utf-8") as file:
    season_data = json.load(file)

season_matches = pd.json_normalize(season_data["matches"])
season_matches["season"] = "2026_27"

# Invalid or missing dates become NaT instead of raising an error.
season_matches["date"] = pd.to_datetime(
    season_matches["date"],
    errors="coerce"
)

# Missing scores are expected for upcoming fixtures.
if "score.ft" not in season_matches.columns:
    season_matches["score.ft"] = None

# Use an alternative score only if it contains two goal totals.
if "score" in season_matches.columns:
    alternative_scores = season_matches["score"].apply(
        lambda score: (
            score
            if isinstance(score, list) and len(score) == 2
            else None
        )
    )

    season_matches["score.ft"] = (
        season_matches["score.ft"].fillna(alternative_scores)
    )

def is_completed_score(score):
    return (
        isinstance(score, list)
        and len(score) == 2
        and all(
            isinstance(goal, (int, float))
            and pd.notna(goal)
            and goal >= 0
            and float(goal).is_integer()
            for goal in score
        )
    )

completed_mask = season_matches["score.ft"].apply(
    is_completed_score
)

# Keep unplayed or incomplete fixtures separate.
pending_matches = season_matches.loc[~completed_mask].copy()

test_matches = (
    season_matches.loc[completed_mask]
    .sort_values(["date", "time"] if "time" in season_matches else ["date"])
    .reset_index(drop=True)
)

test_matches["home_goals"] = test_matches["score.ft"].str[0].astype(int)
test_matches["away_goals"] = test_matches["score.ft"].str[1].astype(int)

### Validate

- Checks that completed matches have dates and team names, are chronological, contain no duplicate fixtures, and have no future-dated results.
- It prints the counts and date range so you can inspect them.

In [3]:
assert len(test_matches) > 0, "No completed matches found."

assert test_matches["date"].notna().all(), (
    "Some scored matches have missing or invalid dates."
)

assert test_matches["date"].is_monotonic_increasing

assert test_matches[["team1", "team2"]].notna().all().all()

assert not season_matches.duplicated(
    ["season", "team1", "team2"]
).any(), "Duplicate fixtures found."

assert test_matches["date"].max() <= pd.Timestamp.today().normalize(), (
    "A future-dated fixture already has a score; inspect it."
)

print("Total fixtures:", len(season_matches))
print("Completed matches:", len(test_matches))
print("Pending/incomplete fixtures:", len(pending_matches))
print("First completed date:", test_matches["date"].min())
print("Latest completed date:", test_matches["date"].max())

test_matches[
    ["date", "team1", "team2", "home_goals", "away_goals"]
].tail(10)

Total fixtures: 380
Completed matches: 50
Pending/incomplete fixtures: 330
First completed date: 2026-08-21 00:00:00
Latest completed date: 2026-09-20 00:00:00


,date,team1,team2,home_goals,away_goals
40,2026-09-18,Brentford FC,Chelsea FC,3,0
41,2026-09-19,Tottenham Hotspur FC,Aston Villa FC,2,3
42,2026-09-19,Everton FC,Ipswich Town FC,1,0
43,2026-09-19,Brighton & Hove Albion FC,Arsenal FC,3,0
44,2026-09-19,Newcastle United FC,Hull City AFC,2,1
45,2026-09-19,Nottingham Forest FC,Coventry City FC,0,1
46,2026-09-20,AFC Bournemouth,Liverpool FC,0,1
47,2026-09-20,Leeds United FC,Crystal Palace FC,0,0
48,2026-09-20,Manchester City FC,Sunderland AFC,5,3
49,2026-09-20,Fulham FC,Manchester United FC,1,1


### Prepare historical features

In [4]:
from soccer_predictor.data import load_matches, TRAINING_SEASONS
from soccer_predictor.features import build_features, INPUT_COLUMNS

# Load only the five historical training seasons.
historical_matches = load_matches(
    DATA_DIRECTORY, TRAINING_SEASONS
)

assert historical_matches["date"].max() < test_matches["date"].min()

# Keep the original historical order, followed by chronological test matches.
all_matches = pd.concat(
    [historical_matches, test_matches],
    ignore_index=True
)

# Record prematch features before each result updates team history.
all_matches = build_features(
    all_matches,
    k=40,
    home_advantage=50
)

# Separate training and test rows again after building features.
training_rows = all_matches["season"].isin(TRAINING_SEASONS)
test_rows = all_matches["season"].eq("2026_27")

X_train = all_matches.loc[training_rows, INPUT_COLUMNS]
y_train = all_matches.loc[training_rows, "result"]

X_test = all_matches.loc[test_rows, INPUT_COLUMNS]
y_test = all_matches.loc[test_rows, "result"]

print("Training features:", X_train.shape)
print("Test features:", X_test.shape)

Training features: (1900, 9)
Test features: (50, 9)


### 1. Dummy Baseline

In [5]:
from sklearn.dummy import DummyClassifier

baseline_model = DummyClassifier(strategy="prior")
baseline_model.fit(X_train, y_train)

baseline_test_probabilities = baseline_model.predict_proba(X_test)

baseline_test_loss = log_loss(
    y_test,
    baseline_test_probabilities,
    labels=[0, 1, 2]
)

print("Baseline test log loss:", baseline_test_loss)

NameError: name 'log_loss' is not defined

### 2. Uncalibrated Logistic Regression

In [ ]:
from sklearn.metrics import log_loss
from soccer_predictor.models import make_logistic_pipeline

# Fit only on the five historical seasons.
logistic_model = make_logistic_pipeline(C=0.1)
logistic_model.fit(X_train, y_train)

# Predict probabilities for the 50 completed test matches.
logistic_test_probabilities = logistic_model.predict_proba(X_test)

logistic_test_loss = log_loss(
    y_test,
    logistic_test_probabilities,
    labels=[0, 1, 2]
)

print("Logistic regression test log loss:", logistic_test_loss)

Logistic regression test log loss: 1.0431274027926614


### 3. Calibrated Logistic Regression

In [ ]:
import numpy as np

from soccer_predictor.validation import split_calibration_indices
from soccer_predictor.models import (
    fit_calibrated_model,
    predict_calibrated,
)

# Positions of historical matches within all_matches.
historical_indices = np.flatnonzero(training_rows.to_numpy())

fit_indices, calibration_indices = split_calibration_indices(
    all_matches,
    historical_indices
)

# These indices refer to all_matches, so use matching X and y.
X_all = all_matches[INPUT_COLUMNS]
y_all = all_matches["result"]

calibration_base_model, calibrator = fit_calibrated_model(
    make_logistic_pipeline(C=0.1),
    X_all,
    y_all,
    fit_indices,
    calibration_indices
)

# Score the base model before adjustment for a direct comparison.
calibration_base_probabilities = (
    calibration_base_model.predict_proba(X_test)
)

calibrated_test_probabilities = predict_calibrated(
    calibration_base_model,
    calibrator,
    X_test
)

calibration_base_test_loss = log_loss(
    y_test,
    calibration_base_probabilities,
    labels=[0, 1, 2]
)

calibrated_test_loss = log_loss(
    y_test,
    calibrated_test_probabilities,
    labels=[0, 1, 2]
)

print("Base model before calibration:", calibration_base_test_loss)
print("Calibrated model:", calibrated_test_loss)

Base model before calibration: 1.05174870368678
Calibrated model: 1.0299796630235227


### 4. XGBoost

In [ ]:
from soccer_predictor.models import make_xgboost_model

xgboost_model = make_xgboost_model(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=1,
    min_child_weight=5,
    reg_lambda=1,
    subsample=1.0,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    gamma=0
)

xgboost_model.fit(X_train, y_train)

xgboost_test_probabilities = xgboost_model.predict_proba(X_test)

xgboost_test_loss = log_loss(
    y_test,
    xgboost_test_probabilities,
    labels=[0, 1, 2]
)

print("XGBoost test log loss:", xgboost_test_loss)

XGBoost test log loss: 1.065769076347351


### Log Loss comparison ranking on first 50 matches of the 2026/2027 Premier League Season

1. Calibrated Logistic Regression: 1.029980
2. Uncalibrated Logistic Regression: 1.043127
3. XGBoost: 1.065769
4. Dummy Baseline: 1.117493

- The calibrated model reduced test log loss by:
    * 7.83% versus the baseline
    * 3.36% versus XGBoost
    * 1.26% versus Uncalibrated Logistic Regression

### Accuracy Score for Reference:

In [ ]:
from sklearn.metrics import accuracy_score

calibrated_test_predictions = (
    calibrator.classes_[
        calibrated_test_probabilities.argmax(axis=1)
    ]
)

print(
    "Test accuracy:",
    accuracy_score(y_test, calibrated_test_predictions)
)

Test accuracy: 0.46


In [ ]:
baseline_predictions = baseline_model.predict(X_test)

print(
    "Baseline accuracy:",
    accuracy_score(y_test, baseline_predictions)
)

Baseline accuracy: 0.36


### Final Fitted Calibrated Logistic Regression Model

In [ ]:
import joblib

MODEL_DIRECTORY = PROJECT_ROOT / "models"
MODEL_DIRECTORY.mkdir(parents=True, exist_ok=True)

model_bundle = {
    "base_model": calibration_base_model,
    "calibrator": calibrator,
    "input_columns": INPUT_COLUMNS.copy(),
    "classes": calibrator.classes_,
    "training_seasons": TRAINING_SEASONS.copy(),
    "training_end_date": historical_matches["date"].max().isoformat(),
    "elo_k": 40,
    "home_advantage": 50,
}

model_path = MODEL_DIRECTORY / "calibrated_logistic.joblib"

joblib.dump(model_bundle, model_path)

print("Saved model:", model_path)

Saved model: C:\Users\arenk\OneDrive\Desktop\soccer-predictor\models\calibrated_logistic.joblib


In [ ]:
from soccer_predictor.models import predict_calibrated

loaded_bundle = joblib.load(model_path)

loaded_probabilities = predict_calibrated(
    loaded_bundle["base_model"],
    loaded_bundle["calibrator"],
    X_test[loaded_bundle["input_columns"]]
)

np.testing.assert_allclose(
    loaded_probabilities,
    calibrated_test_probabilities
)

print("Saved model reproduces the original predictions.")

Saved model reproduces the original predictions.


### Test functions built for future matches:

In [7]:
from soccer_predictor.features import (
    build_team_state,
    make_fixture_features
)

# Replace these with your chosen fixture's date and exact team names.
prediction_date = "2026-10-10"
home_team = "Arsenal FC"
away_team = "Chelsea FC"

import joblib

model_path = PROJECT_ROOT / "models" / "calibrated_logistic.joblib"
loaded_bundle = joblib.load(model_path)

team_state = build_team_state(
    all_matches,
    prediction_date,
    k=loaded_bundle["elo_k"],
    home_advantage=loaded_bundle["home_advantage"]
)

fixture_features = make_fixture_features(
    team_state,
    home_team,
    away_team
)

fixture_features

,home_elo_before,away_elo_before,elo_difference,home_form_5,away_form_5,home_goals_scored_5,home_goals_conceded_5,away_goals_scored_5,away_goals_conceded_5
0,1779.314708,1548.827357,230.487351,2.4,1.4,1.6,0.8,2.0,2.4


In [8]:
from soccer_predictor.models import predict_calibrated

fixture_probabilities = predict_calibrated(
    loaded_bundle["base_model"],
    loaded_bundle["calibrator"],
    fixture_features[loaded_bundle["input_columns"]]
)[0]

outcome_names = {
    0: f"{home_team} win",
    1: "Draw",
    2: f"{away_team} win"
}

for outcome, probability in zip(
    loaded_bundle["classes"],
    fixture_probabilities
):
    print(f"{outcome_names[outcome]}: {probability:.1%}")

Arsenal FC win: 70.5%
Draw: 20.1%
Chelsea FC win: 9.4%


### Test for prediction function for future matches as they occur

In [9]:
from soccer_predictor.predict import predict_fixture

prediction = predict_fixture(
    model_bundle=loaded_bundle,
    completed_matches=all_matches,
    home_team=home_team,
    away_team=away_team,
    prediction_date=prediction_date,
)

for outcome, probability in prediction.items():
    print(f"{outcome}: {probability:.1%}")

home_win: 70.5%
draw: 20.1%
away_win: 9.4%
